# Training window comparison, walk-forward (1y / 2y / 3y, long and wide)

Same models and same comparison as `train_horizon_comparison_non_walk_forward.ipynb`, but instead of ONE 252-bar forecast period at the end of the data, the data is cut into consecutive **252-bar blocks** and every block is forecast by its own model, trained on the 1, 2 or 3 years right before it (walk-forward). Data: `4_ticker_data_2010_2026.csv`.

- Every block = one model, retrained when the block starts. The forecast days of one block become training days of the next one (rolling: the train window always keeps the last 252 / 504 / 756 bars).
- For each training length a **long** model (per-ticker forecasts + EWMA covariance bank) and a **wide** model (portfolio variance directly): six runs.
- The blocks are counted back from the end of the data, so the last block ends on the last day. The longer the training window, the fewer blocks fit in the data.
- Every run gets **one combined residual bank**: the first block's bank as it is (the usual single-calendar start), then only the out-of-sample forecast rows of every later block. The VaR is read off that bank (`wf.create_combined_residuals_bank_and_read_var_df`, the same reading parameters as `calendar.var_from_residual_bank_forecast_period`), and the same fits can be read again any other way.
- Run-to-run comparisons (backtest, level/volatility, exceedance days) use the days **every** run has forecast (the blocks the longest-training run has); a separate table shows every block of every run.

In [ ]:
from itables import show
import numpy as np
import pandas as pd
import data_management as dm
import walk_forward as wf
from core.feature_blocks import FeatureSpec, TickerOneHot, EwmaVariance, CrossSectionVol
from core.targets import SquaredReturn, RatioToBase, LogVariance, AbsReturn
from graph_helpers import (
    plot_var_vs_loss, plot_var_models_vs_loss, plot_models_output_vs_realized_variance, plot_level_and_volatility,
    var_vs_loss_date_picker, var_models_vs_loss_date_picker, plot_exceedance_days_focus,
    plot_cumulative_tick_loss_difference,
)

In [2]:
TICKERS = ['AAPL', 'JPM', 'XOM', 'KO']

WEIGHTS = {
    'AAPL': 0.25,
    'JPM': 0.25,
    'XOM': 0.25,
    'KO': 0.25,
}

df = pd.read_csv(
    './data_general/4_ticker_data_2010_2026.csv',
    parse_dates=['Date'],
    index_col='Date',
)
print(df.index[0].date(), '->', df.index[-1].date(), '|', len(df), 'bars')

2010-01-04 -> 2026-06-30 | 4147 bars


In [ ]:
FORECAST_WINDOW_BARS = 252                            # one model forecasts one block of 252 bars, then the window moves forward by 252 bars
TRAIN_BARS = {'1y': 252, '2y': 504, '3y': 756}        # training window lengths in bars
EXPANDING = False                                     # False: rolling train window (always TRAIN_BARS bars); True: grows from the first train bar
WARMUP_BARS = 30                                      # bars before train_start used only to warm up rolling windows / EWMA

TARGET_SCALE = 10000                                  # the target (r^2) is multiplied by this for training, predictions are divided back
VAR_CONFIDENCE = 0.95                                 # 0.95 -> lower 5% quantile of the standardized residuals
VAR_WINDOW = 252                                      # VaR quantile window: the last N residuals
VAR_MIN_PERIODS = 30                                  # fewest prior residuals a day needs to get a VaR
EWMA_LAM = 0.94                                       # RiskMetrics decay: larger = forgets slower
EWMA_MIN_PERIODS = 30                                 # bars that seed the EWMA covariance

In [ ]:
# Blocks are counted back from the end of the data: as many full FORECAST_WINDOW_BARS blocks as fit after
# (1 return-less first bar + warmup + train window). The price data is cut so the last block ends on the last day.
n = len(df)
calendars = {}
for name, train_bars in TRAIN_BARS.items():
    n_blocks = (n - WARMUP_BARS - 1 - train_bars) // FORECAST_WINDOW_BARS
    if n_blocks < 1:
        raise ValueError(f'{name}: not enough data for one {FORECAST_WINDOW_BARS}-bar block after {train_bars} train bars')
    first_bar = n - (WARMUP_BARS + 1 + train_bars + n_blocks * FORECAST_WINDOW_BARS)
    setting = wf.WalkForwardSetting(
        train_bars=train_bars, forecast_bars=FORECAST_WINDOW_BARS, expanding=EXPANDING, warmup_bars=WARMUP_BARS,
    )
    calendars[name] = setting.make_calendars(df.iloc[first_bar:], TICKERS, WEIGHTS)

pd.DataFrame({
    name: {
        'blocks': len(cals),
        'forecast days': sum(len(c.forecast_dates) for c in cals),
        'first train': f"{cals[0].train_data.index[0].date()} .. {cals[0].train_data.index[-1].date()}",
        'first block forecast': f"{cals[0].forecast_dates[0].date()} .. {cals[0].forecast_dates[-1].date()}",
        'last block forecast': f"{cals[-1].forecast_dates[0].date()} .. {cals[-1].forecast_dates[-1].date()}",
    }
    for name, cals in calendars.items()
})

In [ ]:
# Long format: one model per block. Banks are SINGLE-USE and are created inside train_walk_forward for every block.
# target = what the model predicts (targets.py): SquaredReturn(scale) is r_next^2 * scale, scale only makes the numbers friendlier.
recipe_long = wf.ModelRecipe(
    'long', target=SquaredReturn(TARGET_SCALE), mu_mode='constant', default_mu_hat=0.0,
    ewma_lam=EWMA_LAM, ewma_min_periods=EWMA_MIN_PERIODS,
)
fits_long = {name: wf.train_walk_forward(cals, recipe_long) for name, cals in calendars.items()}

In [ ]:
# Wide format: one model per block predicts the PORTFOLIO variance directly (no per-ticker model, no EWMA bank).
recipe_wide = wf.ModelRecipe('wide', target=SquaredReturn(TARGET_SCALE), mu_mode='constant', default_mu_hat=0.0)
fits_wide = {name: wf.train_walk_forward(cals, recipe_wide) for name, cals in calendars.items()}

In [7]:
# GARCH(1,1) baseline from the `arch` package (garch_models.py), same calendars / blocks / residual sharing / VaR setting.
# wide = ONE GARCH on the portfolio return: the main baseline (the portfolio variance is the real target).
# long = one GARCH per ticker + the same EWMA correlation bank as the long XGBoost: only the per-ticker variance source differs.
GARCH_KWARGS = dict(mean='Zero', vol='GARCH', p=1, q=1, dist='normal')   # passed to arch.arch_model (o=1 would make it GJR)

recipe_garch_wide = wf.ModelRecipe(
    'wide', model_family='garch', model_kwargs=GARCH_KWARGS, mu_mode='constant', default_mu_hat=0.0,
)
recipe_garch_long = wf.ModelRecipe(
    'long', model_family='garch', model_kwargs=GARCH_KWARGS, mu_mode='constant', default_mu_hat=0.0,
    ewma_lam=EWMA_LAM, ewma_min_periods=EWMA_MIN_PERIODS,
)
fits_garch_wide = {name: wf.train_walk_forward(cals, recipe_garch_wide) for name, cals in calendars.items()}
fits_garch_long = {name: wf.train_walk_forward(cals, recipe_garch_long) for name, cals in calendars.items()}

[INFO] wide train: 252 bars -> 252 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 252 bars -> 252 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 252 bars -> 252 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 252 bars -> 252 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 252 bars -> 252 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 252 bars -> 252 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 252 bars -> 252 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 252 bars -> 252 after dropna (0 drop

In [ ]:
# One residual bank per run (the first block's bank + the out-of-sample forecast rows of every later block), keyed
# '<training length>_<model type>' (1y_long, 1y_wide, 1y_garch_wide, 1y_garch_long, 2y_long, ...).
fits_by_run = {}
for name in TRAIN_BARS:
    fits_by_run[f'{name}_long'] = fits_long[name]
    fits_by_run[f'{name}_wide'] = fits_wide[name]
    fits_by_run[f'{name}_garch_wide'] = fits_garch_wide[name]
    fits_by_run[f'{name}_garch_long'] = fits_garch_long[name]

residual_banks = {key: wf.build_combined_residual_bank_WITHOUT_CALC(fits) for key, fits in fits_by_run.items()}   # for the graphs

# The VaR is READ off the run's combined bank, the reading is chosen here: read the same fits again any other way any time.
var_dfs = {
    key: wf.create_combined_residuals_bank_and_read_var_df(
        fits,
        var_confidence=VAR_CONFIDENCE,
        multi_day_var_calculation_window_size=VAR_WINDOW,
        multi_day_var_min_periods=VAR_MIN_PERIODS,
    )
    for key, fits in fits_by_run.items()
}

# distribution of the VaR level (varForecast) per run, over all of the run's blocks
pd.DataFrame({key: v['varForecast'] for key, v in var_dfs.items()}).describe()

In [ ]:
# Every run in one place, keyed '<training length>_<model type>' (1y_long, 1y_wide, ..., 1y_garch_wide, 1y_garch_long, ...)
# residual_bank = the combined bank of the run (for the hover data and the variance graph)
runs = {key: {'var_df': var_dfs[key], 'residual_bank': residual_banks[key]} for key in fits_by_run}

# The runs have different numbers of blocks, but they all end on the last day: the days EVERY run forecast are
# the last n_common ones. Run-to-run comparisons below use only those days (same days, same realized losses).
n_common = min(len(run['var_df']) for run in runs.values())
common_dates = df.index[-n_common:]
for run in runs.values():
    run['common_var_df'] = run['var_df'].loc[common_dates]

# The graphs and the calendar-based tables read one calendar: the common days as its forecast window and the
# bars before them as its train window. Only used for dates, realized variance and the train/forecast bands.
display_train_bars = min(max(TRAIN_BARS.values()), n - n_common - WARMUP_BARS - 1)
display_calendar = dm.DailyGroundTruthCalendar(
    df, TICKERS, WEIGHTS,
    train_start=df.index[n - n_common - display_train_bars], train_end=df.index[n - n_common - 1],
    forecast_start=df.index[n - n_common], forecast_end=df.index[-1],
    warmup_bars=WARMUP_BARS,
)
assert display_calendar.forecast_dates.equals(common_dates)
print(f'common period: {common_dates[0].date()} -> {common_dates[-1].date()} ({n_common} days)')
list(runs)

In [10]:
# Every block of every run, tested on its own (252 days per block, so the tests have some power).
block_rows = []
for key, run in runs.items():
    for block in wf.var_df_windows(run['var_df']):
        result = dm.run_full_backtest_suite_single_window(
            block['forecast_df'], block['window_start'], block['window_end'], VAR_CONFIDENCE, 0.95,
        )
        block_rows.append({
            'run': key, 'block': block['window_id'],
            'start': block['window_start'].date(), 'end': block['window_end'].date(),
            **{k: result[k] for k in (
                'n_days', 'n_exceedances', 'exceedance_rate', 'traffic_light_zone',
                'kupiec_pof_pass', 'christoffersen_ind_pass', 'christoffersen_cc_pass',
            )},
        })
pd.DataFrame(block_rows).set_index(['run', 'block'])

start         end  n_days  n_exceedances  \
run           block                                                  
1y_long       0      2011-06-17  2012-06-15     252             22   
              1      2012-06-18  2013-06-19     252              9   
              2      2013-06-20  2014-06-19     252             10   
              3      2014-06-20  2015-06-19     252             17   
              4      2015-06-22  2016-06-20     252             13   
...                         ...         ...     ...            ...   
3y_garch_long 8      2021-06-23  2022-06-22     252             14   
              9      2022-06-23  2023-06-23     252              8   
              10     2023-06-26  2024-06-25     252             12   
              11     2024-06-26  2025-06-27     252             17   
              12     2025-06-30  2026-06-30     252             18   

                     exceedance_rate  traffic_light_zone  kupiec_pof_pass  \
run           block                                                         
1y_long       0             0.087302                   2            False   
              1             0.035714                   1             True   
              2             0.039683                   1             True   
              3             0.067460                   1             True   
              4             0.051587                   1             True   
...                              ...                 ...              ...   
3y_garch_long 8             0.055556                   1             True   
              9             0.031746                   1             True   
              10            0.047619                   1             True   
              11            0.067460                   1             True   
              12            0.071429                   1             True   

                     christoffersen_ind_pass  christoffersen_cc_pass  
run           block                                                   
1y_long       0                         True                   False  
              1                         True                    True  
              2                         True                    True  
              3                         True                    True  
              4                         True                    True  
...                                      ...                     ...  
3y_garch_long 8                         True                    True  
              9                         True                    True  
              10                        True                    True  
              11                        True                    True  
              12                        True                    True  

[168 rows x 9 columns]

In [12]:
# From here on: the COMMON days only (the last n_common days, forecast by every run).
# Backtest suite, each run as one window. var_confidence must be the one the VaR was produced with.
backtest = pd.DataFrame({
    key: dm.backtest_forecast_period(display_calendar, run['common_var_df'], VAR_CONFIDENCE)
    for key, run in runs.items()
})
backtest

,1y_long,1y_wide,1y_garch_wide,1y_garch_long,2y_long,2y_wide,2y_garch_wide,2y_garch_long,3y_long,3y_wide,3y_garch_wide,3y_garch_long
var_confidence,0.95,0.95,0.95,0.95,0.95,0.95,0.95,0.95,0.95,0.95,0.95,0.95
test_confidence,0.95,0.95,0.95,0.95,0.95,0.95,0.95,0.95,0.95,0.95,0.95,0.95
n_days,3276,3276,3276,3276,3276,3276,3276,3276,3276,3276,3276,3276
n_exceedances,171,189,172,182,178,189,179,183,179,183,173,178
exceedance_rate,0.052198,0.057692,0.052503,0.055556,0.054335,0.057692,0.05464,0.055861,0.05464,0.055861,0.052808,0.054335
expected_exceedance_rate,0.05,0.05,0.05,0.05,0.05,0.05,0.05,0.05,0.05,0.05,0.05,0.05
binomial_pass,True,False,True,True,True,False,True,True,True,True,True,True
traffic_light_zone,1,2,1,1,1,2,1,1,1,1,1,1
kupiec_pof_stat,0.328616,3.89672,0.425441,2.057869,1.261796,3.89672,1.443147,2.286128,1.443147,2.286128,0.534539,1.261796
kupiec_pof_critical_value,3.841459,3.841459,3.841459,3.841459,3.841459,3.841459,3.841459,3.841459,3.841459,3.841459,3.841459,3.841459


In [13]:
# Summary: exceedances, whether the tests passed, and the mean VaR level
summary_rows = [
    'n_exceedances', 'exceedance_rate', 'expected_exceedance_rate', 'traffic_light_zone',
    'kupiec_pof_stat', 'kupiec_pof_critical_value', 'kupiec_pof_pass',
    'christoffersen_ind_pass', 'christoffersen_cc_pass',
]
summary = backtest.loc[summary_rows].copy()
summary.loc['mean_varForecast'] = [runs[key]['common_var_df']['varForecast'].astype(float).mean() for key in summary.columns]
summary

,1y_long,1y_wide,1y_garch_wide,1y_garch_long,2y_long,2y_wide,2y_garch_wide,2y_garch_long,3y_long,3y_wide,3y_garch_wide,3y_garch_long
n_exceedances,171,189,172,182,178,189,179,183,179,183,173,178
exceedance_rate,0.052198,0.057692,0.052503,0.055556,0.054335,0.057692,0.05464,0.055861,0.05464,0.055861,0.052808,0.054335
expected_exceedance_rate,0.05,0.05,0.05,0.05,0.05,0.05,0.05,0.05,0.05,0.05,0.05,0.05
traffic_light_zone,1,2,1,1,1,2,1,1,1,1,1,1
kupiec_pof_stat,0.328616,3.89672,0.425441,2.057869,1.261796,3.89672,1.443147,2.286128,1.443147,2.286128,0.534539,1.261796
kupiec_pof_critical_value,3.841459,3.841459,3.841459,3.841459,3.841459,3.841459,3.841459,3.841459,3.841459,3.841459,3.841459,3.841459
kupiec_pof_pass,True,False,True,True,True,False,True,True,True,True,True,True
christoffersen_ind_pass,False,False,False,True,False,True,False,False,True,False,False,True
christoffersen_cc_pass,True,False,False,True,True,True,False,False,True,False,True,True
mean_varForecast,0.017537,0.018695,0.016492,0.015983,0.016518,0.017657,0.016442,0.015786,0.016165,0.018752,0.016123,0.015807


In [14]:
# Level and volatility suite (descriptive, no hypothesis tests): how HIGH is the VaR and how much does it MOVE.
# mean_tick_loss is the headline score (lower is better) = exceedance part (VaR too low) + overshoot part (VaR too high).
level_volatility = pd.DataFrame({
    key: dm.level_and_volatility_forecast_period(display_calendar, run['common_var_df'], VAR_CONFIDENCE)
    for key, run in runs.items()
})
level_volatility

,1y_long,1y_wide,1y_garch_wide,1y_garch_long,2y_long,2y_wide,2y_garch_wide,2y_garch_long,3y_long,3y_wide,3y_garch_wide,3y_garch_long
var_confidence,0.950000,0.950000,0.950000,0.950000,0.950000,0.950000,0.950000,0.950000,0.950000,0.950000,0.950000,0.950000
n_days,3276.000000,3276.000000,3276.000000,3276.000000,3276.000000,3276.000000,3276.000000,3276.000000,3276.000000,3276.000000,3276.000000,3276.000000
n_exceedances,171.000000,189.000000,172.000000,182.000000,178.000000,189.000000,179.000000,183.000000,179.000000,183.000000,173.000000,178.000000
exceedance_rate,0.052198,0.057692,0.052503,0.055556,0.054335,0.057692,0.054640,0.055861,0.054640,0.055861,0.052808,0.054335
expected_exceedance_rate,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000
mean_tick_loss,0.001331,0.001401,0.001208,0.001193,0.001262,0.001350,0.001187,0.001173,0.001248,0.001432,0.001180,0.001179
tick_loss_exceedance_part,0.000399,0.000411,0.000332,0.000342,0.000382,0.000412,0.000314,0.000333,0.000386,0.000437,0.000322,0.000337
tick_loss_overshoot_part,0.000932,0.000990,0.000876,0.000851,0.000880,0.000939,0.000873,0.000841,0.000863,0.000995,0.000857,0.000842
mean_var,0.017537,0.018695,0.016492,0.015983,0.016518,0.017657,0.016442,0.015786,0.016165,0.018752,0.016123,0.015807
median_var,0.013369,0.014476,0.013476,0.013272,0.013157,0.013372,0.012808,0.013092,0.012938,0.013330,0.012870,0.012910


In [15]:
# Every run against the GARCH wide baseline of the SAME training length (same blocks, same common days).
# Differences are run - baseline, so NEGATIVE = better than GARCH.
#   days_better_share : share of days the run's tick loss is below the baseline's
#   dm_t_stat / dm_p_value : Diebold-Mariano test on the daily tick-loss difference (HAC standard error);
#                            a small p-value = the difference is more than day-to-day noise.
garch_comparison = pd.concat({
    name: dm.compare_to_baseline(
        {key: run['common_var_df'] for key, run in runs.items() if key.startswith(f'{name}_')},
        baseline=f'{name}_garch_wide',
        var_confidence=VAR_CONFIDENCE,
    )
    for name in TRAIN_BARS
})
garch_comparison

mean_tick_loss  tick_loss_diff  tick_loss_relative_diff  \
1y 1y_long              0.001331    1.230532e-04                 0.101879   
   1y_wide              0.001401    1.936022e-04                 0.160288   
   1y_garch_wide        0.001208    0.000000e+00                 0.000000   
   1y_garch_long        0.001193   -1.500723e-05                -0.012425   
2y 2y_long              0.001262    7.559778e-05                 0.063705   
   2y_wide              0.001350    1.635457e-04                 0.137818   
   2y_garch_wide        0.001187    0.000000e+00                 0.000000   
   2y_garch_long        0.001173   -1.324055e-05                -0.011158   
3y 3y_long              0.001248    6.856889e-05                 0.058132   
   3y_wide              0.001432    2.520822e-04                 0.213713   
   3y_garch_wide        0.001180    0.000000e+00                 0.000000   
   3y_garch_long        0.001179   -5.158236e-07                -0.000437   

                  days_better_share  n_exceedances  exceedance_diff  \
1y 1y_long                 0.540293          171.0             -1.0   
   1y_wide                 0.459096          189.0             17.0   
   1y_garch_wide                NaN          172.0              0.0   
   1y_garch_long           0.603175          182.0             10.0   
2y 2y_long                 0.538156          178.0             -1.0   
   2y_wide                 0.504579          189.0             10.0   
   2y_garch_wide                NaN          179.0              0.0   
   2y_garch_long           0.560745          183.0              4.0   
3y 3y_long                 0.558608          179.0              6.0   
   3y_wide                 0.485653          183.0             10.0   
   3y_garch_wide                NaN          173.0              0.0   
   3y_garch_long           0.553724          178.0              5.0   

                  dm_t_stat  dm_p_value  
1y 1y_long         2.349589    0.018794  
   1y_wide         4.369112    0.000012  
   1y_garch_wide        NaN         NaN  
   1y_garch_long  -0.682063    0.495199  
2y 2y_long         1.555991    0.119710  
   2y_wide         3.080207    0.002069  
   2y_garch_wide        NaN         NaN  
   2y_garch_long  -0.761368    0.446437  
3y 3y_long         1.647732    0.099408  
   3y_wide         3.751780    0.000176  
   3y_garch_wide        NaN         NaN  
   3y_garch_long  -0.059395    0.952638

In [16]:
# Where does each run gain or lose against the GARCH wide baseline? Running sum of the daily tick-loss difference:
# a line going DOWN = better than GARCH in that stretch, UP = worse. One figure per training length.
for name in TRAIN_BARS:
    plot_cumulative_tick_loss_difference(
        {key: run['common_var_df'] for key, run in runs.items() if key.startswith(f'{name}_')},
        baseline=f'{name}_garch_wide',
        var_confidence=VAR_CONFIDENCE,
        title=f'Cumulative tick loss difference vs {name}_garch_wide (common days)',
    ).show()

In [17]:
# Is the difference one big block or consistent? For every run: in how many of ITS OWN blocks (252 days each, all the
# blocks of the run, not only the common days) is the mean tick loss below the GARCH wide baseline of the same length?
rows = {}
for name in TRAIN_BARS:
    block_tick = {
        key: pd.Series({
            block['window_id']: dm.daily_tick_loss(block['forecast_df'], VAR_CONFIDENCE).mean()
            for block in wf.var_df_windows(run['var_df'])
        })
        for key, run in runs.items() if key.startswith(f'{name}_')
    }
    baseline = block_tick[f'{name}_garch_wide']
    for key, series in block_tick.items():
        rows[key] = {'blocks': len(series), 'blocks_better_than_garch_wide': int((series < baseline).sum())}
pd.DataFrame(rows).T

,blocks,blocks_better_than_garch_wide
1y_long,15,7
1y_wide,15,2
1y_garch_wide,15,0
1y_garch_long,15,8
2y_long,14,6
2y_wide,14,2
2y_garch_wide,14,0
2y_garch_long,14,8
3y_long,13,4
3y_wide,13,2


In [18]:
# Do the models miss on the same days? One row per day where at least one run has loss > VaR.
# n_models = how many runs exceeded that day (len(runs) = all of them, 1 = only one run).
exceedance_days = dm.exceedance_days_by_model({key: run['common_var_df'] for key, run in runs.items()})
show(exceedance_days)

In [19]:
# How much do the runs agree? Number of days by how many runs missed them.
exceedance_days['n_models'].value_counts().sort_index().rename_axis('runs that missed the day').to_frame('days')

,days
runs that missed the day,
1,59
2,29
3,14
4,11
5,13
6,22
7,12
8,10
9,13


In [20]:
# Per (day, model): realized loss, VaR forecast and their difference (loss - VaR, positive = the VaR was exceeded by that much).
exceedance_details = dm.exceedance_day_details({key: run['common_var_df'] for key, run in runs.items()}, exceedance_days)
show(exceedance_details.reset_index())

In [21]:
# The exceedance days stacked in ONE figure: per day a graph (every run's VaR line + the realized loss) and,
# next to it, a table with loss, VaR and loss - VaR per run (exceeded rows in red).
# The common period is long now, so only the N_FOCUS_DAYS exceedance days with the largest realized loss are drawn.
N_FOCUS_DAYS = 15
focus_days = exceedance_days.sort_values('loss', ascending=False).head(N_FOCUS_DAYS).sort_index()
plot_exceedance_days_focus(
    {key: run['common_var_df'] for key, run in runs.items()},
    focus_days,
    window_bars=10,
)

In [22]:
# The VaR of every run (long and wide, 1y / 2y / 3y) on top of the same realized loss, common days
plot_var_models_vs_loss(
    {key: (run['common_var_df'], run['residual_bank']) for key, run in runs.items()},
    title='VaR Forecast vs Realized Loss (walk-forward, long / wide, 1y / 2y / 3y training)',
    calendar=display_calendar,
)

In [23]:
# Same graph with a date picker (needs ipywidgets): pick a start date and a window length, the graph jumps there.
var_models_vs_loss_date_picker(
    {key: (run['common_var_df'], run['residual_bank']) for key, run in runs.items()},
    title='VaR Forecast vs Realized Loss (walk-forward, long / wide, 1y / 2y / 3y training)',
    calendar=display_calendar,
)

In [24]:
# Model outputs (predicted_variance) against the EWMA realized variance, common days
plot_models_output_vs_realized_variance(
    display_calendar,
    {key: run['residual_bank'] for key, run in runs.items()},
    only_forecast_window=True,
)

In [25]:
# One graph per run over ALL of its blocks (hover shows predicted_variance, predicted_sigma, mu_hat and the standardized residual)
for key, run in runs.items():
    plot_var_vs_loss(run['var_df'], run['residual_bank'], title=f'VaR vs Realized Loss ({key}, walk-forward, all blocks)').show()

## Strategies: feature sets and targets

A strategy is a `ModelRecipe`: a `FeatureSpec` (which feature blocks the model sees) and a `Target` (what it is trained to predict, see `targets.py`). Long and wide come from the same blocks.

| Target | The model predicts | Variance = |
|---|---|---|
| `SquaredReturn(scale)` | next-bar r² × scale | prediction / scale |
| `RatioToBase(clip)` | r² / EWMA variance (a correction to the EWMA, 1 = "the EWMA is right") | prediction × EWMA variance |
| `LogVariance()` | log(r²) | exp(prediction) |
| `AbsReturn()` | \|r\| | prediction² |

The cells below train a few of them for ONE training length on the same blocks as above and compare them with the pure EWMA (no model, the "do nothing" baseline) and the GARCH wide baseline on the common days.

In [ ]:
STRATEGY_TRAIN = '3y'                                 # which training length the strategies are tried on (a key of TRAIN_BARS)
RATIO_CLIP = 20.0                                     # r^2 / EWMA variance is capped at this for TRAINING only (heavy tail)

# feature sets: the default blocks, + a ticker identity (long only), + the EWMA variance and the other tickers' vol
with_ticker = FeatureSpec.default().with_blocks(TickerOneHot())
with_extras = FeatureSpec.default().with_blocks(TickerOneHot(), EwmaVariance(EWMA_LAM), CrossSectionVol(21))

strategy_recipes = {
    # 'ewma' = no model at all: the pure RiskMetrics EWMA variance, read like any model
    'ewma':                wf.ModelRecipe('wide', model_family='ewma', ewma_lam=EWMA_LAM, ewma_min_periods=EWMA_MIN_PERIODS),
    'long_ratio':          wf.ModelRecipe('long', target=RatioToBase(clip=RATIO_CLIP), ewma_lam=EWMA_LAM, ewma_min_periods=EWMA_MIN_PERIODS),
    'long_ratio_ticker':   wf.ModelRecipe('long', target=RatioToBase(clip=RATIO_CLIP), features=with_ticker, ewma_lam=EWMA_LAM, ewma_min_periods=EWMA_MIN_PERIODS),
    'long_ratio_extras':   wf.ModelRecipe('long', target=RatioToBase(clip=RATIO_CLIP), features=with_extras, ewma_lam=EWMA_LAM, ewma_min_periods=EWMA_MIN_PERIODS),
    'long_log':            wf.ModelRecipe('long', target=LogVariance(), ewma_lam=EWMA_LAM, ewma_min_periods=EWMA_MIN_PERIODS),
    'wide_ratio':          wf.ModelRecipe('wide', target=RatioToBase(clip=RATIO_CLIP)),
    'wide_log':            wf.ModelRecipe('wide', target=LogVariance()),
}

strategy_var_dfs = {}
for name, recipe in strategy_recipes.items():
    fits = wf.train_walk_forward(calendars[STRATEGY_TRAIN], recipe)
    strategy_var_dfs[name] = wf.create_combined_residuals_bank_and_read_var_df(
        fits,
        var_confidence=VAR_CONFIDENCE,
        multi_day_var_calculation_window_size=VAR_WINDOW,
        multi_day_var_min_periods=VAR_MIN_PERIODS,
    ).loc[common_dates]

# the runs trained above, for the same training length (squared-return target, GARCH baseline)
for model_type in ('long', 'wide', 'garch_wide'):
    strategy_var_dfs[f'{model_type}_squared' if model_type != 'garch_wide' else 'garch_wide'] = runs[f'{STRATEGY_TRAIN}_{model_type}']['common_var_df']
list(strategy_var_dfs)

In [ ]:
# Every strategy against the pure EWMA (no model) on the common days. Differences are strategy - baseline, so
# NEGATIVE = better than the EWMA; dm_p_value = Diebold-Mariano test of the daily tick-loss difference.
strategy_comparison = dm.compare_to_baseline(strategy_var_dfs, baseline='ewma', var_confidence=VAR_CONFIDENCE)
strategy_comparison.sort_values('mean_tick_loss')

In [ ]:
# Where does each strategy gain or lose against the pure EWMA? (running sum of the daily tick-loss difference,
# below 0 = better than the EWMA in that stretch)
plot_cumulative_tick_loss_difference(
    strategy_var_dfs, baseline='ewma', var_confidence=VAR_CONFIDENCE,
    title=f'Cumulative tick loss difference vs pure EWMA ({STRATEGY_TRAIN} training, common days)',
)